# Day 084 — Exercise 3: Handoffs — Explicit Data Passing

**What you'll build:** the `Handoff` dataclass and `summarize_handoffs` — a record of what was passed from one agent to the next, and a way to render the full handoff chain at a glance.

**Why it matters:** in a multi-agent system, data flows from agent to agent. Without explicit handoffs, that flow is invisible — you have to guess what the researcher passed to the writer. A `Handoff` makes it an auditable record: who sent what to whom, and with what metadata. `summarize_handoffs` turns that record into a one-glance debug trace, analogous to Day 80's `trace` on the ReAct agent.

In [ ]:
def _mock_multi(findings='Finding: AI agents collaborate.', document='Doc: Agents work together.'):
    """Branch on system message: 'research specialist' -> findings, else -> document."""
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'research specialist' in system.lower():
            return findings
        return document
    return _fn
import json

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── researcher specialist ─────────────────────────────────────────────────────
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    system = "\n".join([
        "You are a research specialist. Your job is to gather relevant facts and",
        "key information about the topic given to you.",
        "",
        "Return a structured list of the most important findings.",
        "Be factual, concise, and cover the main points.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": "Research topic: " + str(query)}]


class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings.

    Each call to research() returns a string of findings and records the
    exchange in history. The agent has one job: gather facts. It passes its
    output to the next agent via a Handoff — it does not write, review, or
    plan.

    Example::

        researcher = ResearcherAgent(llm_fn=my_llm_fn)
        findings = researcher.research("topological sort algorithms")
    """

    def __init__(self, llm_fn=None):
        self._llm_fn = llm_fn
        self._history = []

    def research(self, query):
        """Research a query and return findings as a string."""
        messages = build_researcher_prompt(query)
        findings = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"query": query, "findings": findings})
        return findings

    def history(self):
        """Return a copy of the research history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()

# ── writer specialist ─────────────────────────────────────────────────────────
def build_writer_prompt(findings, style="concise", instructions=None):
    """Build a prompt for the writer role: turn findings into a document."""
    system_parts = [
        "You are a writing specialist. Turn the provided findings into",
        "a polished, well-structured document.",
        "Style: " + str(style) + ".",
    ]
    if instructions:
        system_parts.append("Additional instructions: " + str(instructions))
    system = "\n".join(system_parts)
    user = "Findings:\n" + str(findings)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


class WriterAgent:
    """A specialist that turns research findings into a polished document.

    The writer has one job: take findings (a string from the ResearcherAgent
    or any other source) and produce a well-structured document. The style
    controls the tone (e.g. 'concise', 'detailed', 'formal').

    Example::

        writer = WriterAgent(llm_fn=my_llm_fn, style="concise")
        document = writer.write(findings)
    """

    def __init__(self, llm_fn=None, style="concise"):
        self._llm_fn = llm_fn
        self.style = style
        self._history = []

    def write(self, findings, instructions=None):
        """Write a document from findings. Returns the document string."""
        messages = build_writer_prompt(findings, style=self.style,
                                       instructions=instructions)
        document = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"findings": findings, "document": document})
        return document

    def history(self):
        """Return a copy of the writing history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()


## Task

1. `Handoff` dataclass — four fields: `from_agent: str`, `to_agent: str`, `content: str`, `metadata: dict = field(default_factory=dict)`.
2. `summarize_handoffs(handoffs) -> str` — one line per handoff: `'from -> to: <first 80 chars of content>'`. Return `''` for an empty list.

## Your Implementation

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Handoff:
    """An explicit record of data passed from one agent to the next."""
    from_agent: str = ''      # TODO: correct type + all four fields
    to_agent: str = ''
    content: str = ''

def summarize_handoffs(handoffs):
    """Render the handoff chain as text: one 'from -> to: preview' line each."""
    raise NotImplementedError


In [ ]:

# ── handoffs: explicit data passing between agents ────────────────────────────
from dataclasses import dataclass, field


@dataclass
class Handoff:
    """An explicit record of data passed from one agent to the next.

    Attributes:
        from_agent: name of the sending agent ('researcher', 'writer', ...).
        to_agent:   name of the receiving agent.
        content:    the data being passed (findings string, document, ...).
        metadata:   optional dict for any extra context (topic, style, ...).
    """
    from_agent: str
    to_agent: str
    content: str
    metadata: dict = field(default_factory=dict)


def summarize_handoffs(handoffs):
    """Render the handoff chain as a human-readable text for debugging.

    Each line shows the sender, receiver, and the first 80 characters of the
    content, so you can see the full flow at a glance.
    """
    lines = []
    for h in handoffs:
        preview = h.content[:80].replace("\n", " ")
        lines.append(h.from_agent + " -> " + h.to_agent + ": " + preview)
    return "\n".join(lines)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    h = Handoff('researcher', 'writer', 'here are the facts')
    assert h.from_agent == 'researcher' and h.to_agent == 'writer'
    assert h.content == 'here are the facts'
    score += 1; print("✅ Handoff stores from_agent, to_agent, content")

    assert h.metadata == {}
    h2 = Handoff('a', 'b', 'x', metadata={'key': 'val'})
    assert h2.metadata == {'key': 'val'}
    score += 1; print("✅ metadata defaults to {} and can be set")

    summary = summarize_handoffs([h, h2])
    assert 'researcher -> writer' in summary and 'a -> b' in summary
    score += 1; print("✅ summarize_handoffs renders the chain")

    long_content = 'x' * 200
    h_long = Handoff('a', 'b', long_content)
    line = summarize_handoffs([h_long]).splitlines()[0]
    assert len(line) < 200           # preview is capped, not the full content
    score += 1; print("✅ summarize_handoffs truncates long content to a preview")

    assert summarize_handoffs([]) == ''
    score += 1; print("✅ summarize_handoffs handles an empty list")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── handoffs: explicit data passing between agents ────────────────────────────
from dataclasses import dataclass, field


@dataclass
class Handoff:
    """An explicit record of data passed from one agent to the next.

    Attributes:
        from_agent: name of the sending agent ('researcher', 'writer', ...).
        to_agent:   name of the receiving agent.
        content:    the data being passed (findings string, document, ...).
        metadata:   optional dict for any extra context (topic, style, ...).
    """
    from_agent: str
    to_agent: str
    content: str
    metadata: dict = field(default_factory=dict)


def summarize_handoffs(handoffs):
    """Render the handoff chain as a human-readable text for debugging.

    Each line shows the sender, receiver, and the first 80 characters of the
    content, so you can see the full flow at a glance.
    """
    lines = []
    for h in handoffs:
        preview = h.content[:80].replace("\n", " ")
        lines.append(h.from_agent + " -> " + h.to_agent + ": " + preview)
    return "\n".join(lines)
```

**Why make handoffs explicit instead of just chaining function calls?** An explicit `Handoff` is an audit trail. When a multi-agent run goes wrong, you want to know exactly what the researcher sent to the writer — not just that something went wrong in the middle. `summarize_handoffs` gives you that trace in one print call.

</details>